In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("/content/drive/MyDrive/MVJ_Internship/Data")

file_1 = data_dir / "online_retail_II_csv(2009-2010).csv"
file_2 = data_dir / "online_retail_II_csv(2010-2011).csv"

print("First file exists:", file_1.exists())
print("Second file exists:", file_2.exists())

Mounted at /content/drive
First file exists: True
Second file exists: True


In [3]:
#load the original data safely using ISO-8859-1 encoding to handle special characters (e.g. £)
df_1 = pd.read_csv(
    file_1,
    low_memory=False,
    dtype={"Invoice": "string", "StockCode": "string"},
    encoding="ISO-8859-1"
)

df_2 = pd.read_csv(
    file_2,
    low_memory=False,
    dtype={"Invoice": "string", "StockCode": "string"},
    encoding="ISO-8859-1"
)

raw_df = pd.concat([df_1, df_2], ignore_index=True)

print("First file rows:", len(df_1))
print("Second file rows:", len(df_2))
print("Combined rows:", len(raw_df))

display(raw_df.head())

First file rows: 525461
Second file rows: 541910
Combined rows: 1067371


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01-12-09,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01-12-09,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01-12-09,1.25,13085.0,United Kingdom


In [4]:
#incspect column and data types
print("Dataset shape:", raw_df.shape)

print("\nColumn names:")
print(raw_df.columns.tolist())

print("\nData types:")
print(raw_df.dtypes)

print("\nMissing values per column:")
display(raw_df.isna().sum().to_frame("Missing_Count"))

print("\nSample rows:")
display(raw_df.head(10))

Dataset shape: (1067371, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice        string[python]
StockCode      string[python]
Description            object
Quantity                int64
InvoiceDate            object
Price                 float64
Customer ID           float64
Country                object
dtype: object

Missing values per column:


,Missing_Count
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0



Sample rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01-12-09,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01-12-09,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01-12-09,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,01-12-09,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,01-12-09,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,01-12-09,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,01-12-09,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,01-12-09,3.75,13085.0,United Kingdom


In [5]:
# Remove accidental spaces from column names
raw_df.columns = raw_df.columns.str.strip()

# Work on a separate copy
clean_df = raw_df.copy()

# Convert date and numeric columns
clean_df["InvoiceDate"] = pd.to_datetime(
    clean_df["InvoiceDate"], errors="coerce", dayfirst=True
)

clean_df["Quantity"] = pd.to_numeric(
    clean_df["Quantity"], errors="coerce"
)

clean_df["Price"] = pd.to_numeric(
    clean_df["Price"], errors="coerce"
)

# Standardize invoice text
clean_df["Invoice"] = clean_df["Invoice"].astype("string").str.strip()

# Create a revenue column
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["Price"]

print("Rows:", len(clean_df))
print("Invalid dates:", clean_df["InvoiceDate"].isna().sum())
print("Invalid quantities:", clean_df["Quantity"].isna().sum())
print("Invalid prices:", clean_df["Price"].isna().sum())

display(clean_df.head())

/tmp/ipykernel_1831/2443956090.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  clean_df["InvoiceDate"] = pd.to_datetime(


Rows: 1067371
Invalid dates: 0
Invalid quantities: 0
Invalid prices: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,30.0


In [6]:
#identify cancellations
clean_df["Is_Cancellation_Invoice"] = (
    clean_df["Invoice"]
    .str.upper()
    .str.startswith("C", na=False)
)

cancellations = clean_df[
    clean_df["Is_Cancellation_Invoice"]
].copy()

print("Cancellation rows:", len(cancellations))
print(
    "Cancellation invoice count:",
    cancellations["Invoice"].nunique()
)

display(
    cancellations[
        ["Invoice", "InvoiceDate", "Quantity", "Price", "Revenue"]
    ].head(10)
)

Cancellation rows: 19494
Cancellation invoice count: 8292


,Invoice,InvoiceDate,Quantity,Price,Revenue
178,C489449,2009-12-01,-12,2.95,-35.40
179,C489449,2009-12-01,-6,1.65,-9.90
180,C489449,2009-12-01,-4,4.25,-17.00
181,C489449,2009-12-01,-6,2.10,-12.60
182,C489449,2009-12-01,-12,2.95,-35.40
183,C489449,2009-12-01,-12,1.25,-15.00
184,C489449,2009-12-01,-12,1.25,-15.00
185,C489449,2009-12-01,-24,0.85,-20.40
186,C489449,2009-12-01,-12,2.95,-35.40
196,C489459,2009-12-01,-3,4.25,-12.75


In [7]:
#identify returns and negative quantities
clean_df["Is_Negative_Quantity"] = clean_df["Quantity"].lt(0)

negative_quantity_rows = clean_df[
    clean_df["Is_Negative_Quantity"]
].copy()

print("Rows with negative quantity:", len(negative_quantity_rows))

print(
    "Negative-quantity rows with cancellation invoices:",
    negative_quantity_rows["Is_Cancellation_Invoice"].sum()
)

print(
    "Negative-quantity rows without cancellation invoices:",
    (
        ~negative_quantity_rows["Is_Cancellation_Invoice"]
    ).sum()
)

display(
    negative_quantity_rows[
        ["Invoice", "InvoiceDate", "Quantity", "Price", "Revenue"]
    ].head(10)
)

Rows with negative quantity: 22950
Negative-quantity rows with cancellation invoices: 19493
Negative-quantity rows without cancellation invoices: 3457


,Invoice,InvoiceDate,Quantity,Price,Revenue
178,C489449,2009-12-01,-12,2.95,-35.40
179,C489449,2009-12-01,-6,1.65,-9.90
180,C489449,2009-12-01,-4,4.25,-17.00
181,C489449,2009-12-01,-6,2.10,-12.60
182,C489449,2009-12-01,-12,2.95,-35.40
183,C489449,2009-12-01,-12,1.25,-15.00
184,C489449,2009-12-01,-12,1.25,-15.00
185,C489449,2009-12-01,-24,0.85,-20.40
186,C489449,2009-12-01,-12,2.95,-35.40
196,C489459,2009-12-01,-3,4.25,-12.75


In [8]:
#identify missing customer ids
customer_col = "Customer ID"

if customer_col not in clean_df.columns:
    raise KeyError(
        f"Expected column '{customer_col}' not found. "
        f"Available columns: {clean_df.columns.tolist()}"
    )

missing_customer_rows = clean_df[
    clean_df[customer_col].isna()
].copy()

print("Rows with missing customer IDs:", len(missing_customer_rows))
print(
    "Percentage missing:",
    round(clean_df[customer_col].isna().mean() * 100, 2),
    "%"
)

display(missing_customer_rows.head())

Rows with missing customer IDs: 243007
Percentage missing: 22.77 %


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity
263,489464,21733,85123a mixed,-96,2009-12-01,0.00,NaN,United Kingdom,-0.00,False,True
283,489463,71477,short,-240,2009-12-01,0.00,NaN,United Kingdom,-0.00,False,True
284,489467,85123A,21733 mixed,-192,2009-12-01,0.00,NaN,United Kingdom,-0.00,False,True
470,489521,21646,NaN,-50,2009-12-01,0.00,NaN,United Kingdom,-0.00,False,True
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01,0.55,NaN,United Kingdom,0.55,False,False


In [9]:
#detect duplicate rows
duplicate_count = clean_df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

duplicates = clean_df[
    clean_df.duplicated(keep=False)
].sort_values(["Invoice", "StockCode"])

display(duplicates.head(20))

Exact duplicate rows: 34337


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,5.10,False,False


In [10]:
#detect outliers using the IQR method
# Use positive quantities and prices for sales-outlier analysis
positive_sales = clean_df[
    ~clean_df["Is_Cancellation_Invoice"]
    & clean_df["Quantity"].gt(0)
    & clean_df["Price"].gt(0)
    & clean_df["Revenue"].notna()
].copy()

q1 = positive_sales["Revenue"].quantile(0.25)
q3 = positive_sales["Revenue"].quantile(0.75)

iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

positive_sales["Is_Revenue_Outlier"] = (
    (positive_sales["Revenue"] < lower_bound)
    | (positive_sales["Revenue"] > upper_bound)
)

outliers = positive_sales[
    positive_sales["Is_Revenue_Outlier"]
].copy()

print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))
print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Potential outlier rows:", len(outliers))

display(
    outliers[
        ["Invoice", "InvoiceDate", "StockCode", "Quantity", "Price", "Revenue"]
    ].sort_values("Revenue", ascending=False).head(20)
)

Q1: 3.9
Q3: 17.7
IQR: 13.8
Lower bound: -16.8
Upper bound: 38.4
Potential outlier rows: 85726


,Invoice,InvoiceDate,StockCode,Quantity,Price,Revenue
1065882,581483,2011-12-09,23843,80995,2.08,168469.60
587080,541431,2011-01-18,23166,74215,1.04,77183.60
748132,556444,2011-06-10,22502,60,649.50,38970.00
241827,512771,2010-06-17,M,1,25111.09,25111.09
432176,530715,2010-11-04,84347,9360,1.69,15818.40
540478,537632,2010-12-07,AMAZONFEE,1,13541.33,13541.33
517955,537632,2010-12-07,AMAZONFEE,1,13541.33,13541.33
825443,A563185,2011-08-12,B,1,11062.06,11062.06
135013,502263,2010-03-23,M,1,10953.50,10953.50
135015,502265,2010-03-23,M,1,10953.50,10953.50


In [11]:
#create the cleaned sales dataset
cleaned_sales = clean_df[
    clean_df["InvoiceDate"].notna()
    & clean_df["Quantity"].gt(0)
    & clean_df["Price"].gt(0)
    & clean_df["Revenue"].notna()
    & ~clean_df["Is_Cancellation_Invoice"]
].copy()

# Flag unusually high revenue values without deleting them
cleaned_sales["Is_Revenue_Outlier"] = (
    (cleaned_sales["Revenue"] < lower_bound)
    | (cleaned_sales["Revenue"] > upper_bound)
)

print("Original combined rows:", len(raw_df))
print("Rows in cleaned sales dataset:", len(cleaned_sales))
print("Rows excluded:", len(raw_df) - len(cleaned_sales))
print(
    "Missing customer IDs retained:",
    cleaned_sales[customer_col].isna().sum()
)
print(
    "Outliers flagged:",
    cleaned_sales["Is_Revenue_Outlier"].sum()
)

display(cleaned_sales.head())

Original combined rows: 1067371
Rows in cleaned sales dataset: 1041670
Rows excluded: 25701
Missing customer IDs retained: 236121
Outliers flagged: 85726


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity,Is_Revenue_Outlier
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085.0,United Kingdom,83.4,False,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,81.0,False,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,81.0,False,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,100.8,False,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,30.0,False,False,False


In [12]:
#create the cleaning log
cleaning_log = pd.DataFrame([
    {
        "Issue": "Cancellation invoices",
        "Detection": "Invoice starts with C",
        "Action": "Excluded from cleaned positive-sales dataset",
        "Reason": "Avoid counting cancellation invoices as positive sales"
    },
    {
        "Issue": "Returns / negative quantities",
        "Detection": "Quantity < 0",
        "Action": "Identified and excluded from positive-sales dataset",
        "Reason": "Keep returns separate from positive sales"
    },
    {
        "Issue": "Missing customer IDs",
        "Detection": "Customer ID is missing",
        "Action": "Retained; excluded only from customer-specific calculations when needed",
        "Reason": "Transaction may still contain valid sales information"
    },
    {
        "Issue": "Invalid dates",
        "Detection": "InvoiceDate could not be parsed",
        "Action": "Excluded from cleaned time-series sales dataset",
        "Reason": "Cannot reliably assign transaction to a date"
    },
    {
        "Issue": "Invalid quantities or prices",
        "Detection": "Quantity or Price is missing or non-positive",
        "Action": "Excluded from cleaned positive-sales dataset",
        "Reason": "Not suitable for positive-sales calculations"
    },
    {
        "Issue": "Potential revenue outliers",
        "Detection": "Revenue outside the 1.5 × IQR thresholds",
        "Action": "Flagged for review; not automatically deleted",
        "Reason": "Large transactions may be genuine"
    }
])

display(cleaning_log)

,Issue,Detection,Action,Reason
0,Cancellation invoices,Invoice starts with C,Excluded from cleaned positive-sales dataset,Avoid counting cancellation invoices as positi...
1,Returns / negative quantities,Quantity < 0,Identified and excluded from positive-sales da...,Keep returns separate from positive sales
2,Missing customer IDs,Customer ID is missing,Retained; excluded only from customer-specific...,Transaction may still contain valid sales info...
3,Invalid dates,InvoiceDate could not be parsed,Excluded from cleaned time-series sales dataset,Cannot reliably assign transaction to a date
4,Invalid quantities or prices,Quantity or Price is missing or non-positive,Excluded from cleaned positive-sales dataset,Not suitable for positive-sales calculations
5,Potential revenue outliers,Revenue outside the 1.5 × IQR thresholds,Flagged for review; not automatically deleted,Large transactions may be genuine


In [13]:
#add actual counts to the cleaning log
cleaning_summary = pd.DataFrame([
    {"Metric": "Original combined rows", "Count": len(raw_df)},
    {"Metric": "Cancellation invoice rows", "Count": int(clean_df["Is_Cancellation_Invoice"].sum())},
    {"Metric": "Negative quantity rows", "Count": int(clean_df["Is_Negative_Quantity"].sum())},
    {"Metric": "Missing customer ID rows", "Count": int(clean_df[customer_col].isna().sum())},
    {"Metric": "Exact duplicate rows", "Count": int(raw_df.duplicated().sum())},
    {"Metric": "Invalid date rows", "Count": int(clean_df["InvoiceDate"].isna().sum())},
    {"Metric": "Missing quantity rows", "Count": int(clean_df["Quantity"].isna().sum())},
    {"Metric": "Missing price rows", "Count": int(clean_df["Price"].isna().sum())},
    {"Metric": "Potential revenue outlier rows", "Count": int(outliers.shape[0])},
    {"Metric": "Final cleaned sales rows", "Count": len(cleaned_sales)}
])

display(cleaning_summary)

,Metric,Count
0,Original combined rows,1067371
1,Cancellation invoice rows,19494
2,Negative quantity rows,22950
3,Missing customer ID rows,243007
4,Exact duplicate rows,12135
5,Invalid date rows,0
6,Missing quantity rows,0
7,Missing price rows,0
8,Potential revenue outlier rows,85726
9,Final cleaned sales rows,1041670


In [15]:
# Build five automated data-quality checks
quality_checks = pd.DataFrame([
    {
        "Check": "1. Completeness",
        "Test": "No missing dates in cleaned sales",
        "Passed": bool(cleaned_sales["InvoiceDate"].notna().all()),
        "Details": f"Missing dates: {cleaned_sales['InvoiceDate'].isna().sum()}"
    },
    {
        "Check": "2. Uniqueness",
        "Test": "No exact duplicate rows in cleaned sales",
        "Passed": bool(cleaned_sales.duplicated().sum() == 0),
        "Details": f"Duplicate rows: {cleaned_sales.duplicated().sum()}"
    },
    {
        "Check": "3. Validity",
        "Test": "All quantities and prices are positive",
        "Passed": bool(
            cleaned_sales["Quantity"].gt(0).all()
            and cleaned_sales["Price"].gt(0).all()
        ),
        "Details": (
            f"Non-positive quantities: {(cleaned_sales['Quantity'] <= 0).sum()}; "
            f"non-positive prices: {(cleaned_sales['Price'] <= 0).sum()}"
        )
    },
    {
        "Check": "4. Consistency",
        "Test": "Revenue equals quantity multiplied by price",
        "Passed": bool(
            np.allclose(
                cleaned_sales["Revenue"],
                cleaned_sales["Quantity"] * cleaned_sales["Price"],
                equal_nan=False
            )
        ),
        "Details": "Compares stored revenue with Quantity × Price"
    },
    {
        "Check": "5. Freshness",
        "Test": "Date range is valid and ordered",
        "Passed": bool(
            len(cleaned_sales) > 0
            and cleaned_sales["InvoiceDate"].min()
                <= cleaned_sales["InvoiceDate"].max()
        ),
        "Details": (
            f"From {cleaned_sales['InvoiceDate'].min()} "
            f"to {cleaned_sales['InvoiceDate'].max()}"
        )
    }
])

display(quality_checks)

,Check,Test,Passed,Details
0,1. Completeness,No missing dates in cleaned sales,True,Missing dates: 0
1,2. Uniqueness,No exact duplicate rows in cleaned sales,False,Duplicate rows: 33759
2,3. Validity,All quantities and prices are positive,True,Non-positive quantities: 0; non-positive price...
3,4. Consistency,Revenue equals quantity multiplied by price,True,Compares stored revenue with Quantity × Price
4,5. Freshness,Date range is valid and ordered,True,From 2009-12-01 00:00:00 to 2011-12-09 00:00:00


In [16]:
# Make the quality checks stop the workflow if a test fails
failed_checks = quality_checks[~quality_checks["Passed"]]

print("Total checks:", len(quality_checks))
print("Passed:", int(quality_checks["Passed"].sum()))
print("Failed:", len(failed_checks))

if len(failed_checks) > 0:
    print("\nChecks needing investigation:")
    display(failed_checks)
else:
    print("\nAll five automated quality checks passed.")

Total checks: 5
Passed: 4
Failed: 1

Checks needing investigation:


,Check,Test,Passed,Details
1,2. Uniqueness,No exact duplicate rows in cleaned sales,False,Duplicate rows: 33759


Debugging the failed check

In [17]:
# Count and inspect duplicate rows
# Count exact duplicate rows in the cleaned dataset
duplicate_count = cleaned_sales.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)
print("Total cleaned rows:", len(cleaned_sales))

# Display all copies of duplicate rows
duplicate_rows = cleaned_sales[
    cleaned_sales.duplicated(keep=False)
].sort_values(["Invoice", "StockCode", "InvoiceDate"])

print("Rows involved in duplicate groups:", len(duplicate_rows))

display(duplicate_rows.head(20))

Exact duplicate rows: 33759
Total cleaned rows: 1041670
Rows involved in duplicate groups: 66098


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity,Is_Revenue_Outlier
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,5.10,False,False,False


In [18]:
# Find out which columns are identical
# Check whether duplicate rows agree across the original dataset columns
original_columns = raw_df.columns.tolist()

original_duplicate_count = raw_df.duplicated(
    subset=original_columns
).sum()

print("Exact duplicates in original data:", original_duplicate_count)

# Display duplicate records using only original columns
display(
    raw_df[
        raw_df.duplicated(subset=original_columns, keep=False)
    ].sort_values(["Invoice", "StockCode"]).head(20)
)

Exact duplicates in original data: 12135


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,01-12-09,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,01-12-09,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,01-12-09,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,01-12-09,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,01-12-09,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,01-12-09,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,01-12-09,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,01-12-09,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,01-12-09,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,01-12-09,0.85,16329.0,United Kingdom


In [19]:
# Check whether duplicates are legitimate repeated transactions
inspection_columns = [
    "Invoice",
    "StockCode",
    "InvoiceDate",
    "Quantity",
    "Price",
    "Revenue"
]

display(duplicate_rows[inspection_columns].head(30))

,Invoice,StockCode,InvoiceDate,Quantity,Price,Revenue
379,489517,21491,2009-12-01,1,1.95,1.95
391,489517,21491,2009-12-01,1,1.95,1.95
365,489517,21821,2009-12-01,1,3.75,3.75
386,489517,21821,2009-12-01,1,3.75,3.75
363,489517,21912,2009-12-01,1,3.75,3.75
371,489517,21912,2009-12-01,1,3.75,3.75
394,489517,21912,2009-12-01,1,3.75,3.75
362,489517,21913,2009-12-01,1,3.75,3.75
385,489517,21913,2009-12-01,1,3.75,3.75
368,489517,22130,2009-12-01,6,0.85,5.10


In [20]:
# Check the column names and data

print("Raw data columns:")
print(raw_df.columns.tolist())

print("\nCleaned sales columns:")
print(cleaned_sales.columns.tolist())

print("\nFirst 5 rows of cleaned sales:")
display(cleaned_sales.head())

Raw data columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Cleaned sales columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'Is_Cancellation_Invoice', 'Is_Negative_Quantity', 'Is_Revenue_Outlier']

First 5 rows of cleaned sales:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity,Is_Revenue_Outlier
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085.0,United Kingdom,83.4,False,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,81.0,False,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,81.0,False,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,100.8,False,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,30.0,False,False,False


In [21]:
# Inspect the exact duplicate groups

duplicate_count = cleaned_sales.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

duplicate_rows = cleaned_sales[
    cleaned_sales.duplicated(keep=False)
].sort_values(["Invoice", "StockCode", "InvoiceDate"])

print("Rows involved in duplicate groups:", len(duplicate_rows))

display(duplicate_rows.head(20))

Exact duplicate rows: 33759
Rows involved in duplicate groups: 66098


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity,Is_Revenue_Outlier
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,5.10,False,False,False


In [22]:
#Check whether the source data contains these duplicates
print("Exact duplicates in original data:", raw_df.duplicated().sum())

display(
    raw_df[
        raw_df.duplicated(keep=False)
    ].head(20)
)

Exact duplicates in original data: 12135


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,01-12-09,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,01-12-09,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,01-12-09,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,01-12-09,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,01-12-09,0.85,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,01-12-09,3.75,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,01-12-09,1.95,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,01-12-09,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,01-12-09,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,01-12-09,3.75,16329.0,United Kingdom


In [23]:
#Identify exactly which rows should be reviewed

# Original columns from the source dataset
original_columns = raw_df.columns.tolist()

# Find repeated records in the current cleaned sales dataset
duplicate_mask = cleaned_sales.duplicated(
    subset=original_columns,
    keep=False
)

duplicate_review = cleaned_sales.loc[duplicate_mask].copy()

print("Rows in duplicate groups:", len(duplicate_review))
print(
    "Extra copies beyond the first occurrence:",
    int(cleaned_sales.duplicated(
        subset=original_columns,
        keep="first"
    ).sum())
)

display(duplicate_review.head(20))

Rows in duplicate groups: 66098
Extra copies beyond the first occurrence: 33759


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Cancellation_Invoice,Is_Negative_Quantity,Is_Revenue_Outlier
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01,0.65,16329.0,United Kingdom,7.80,False,False,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,5.10,False,False,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01,1.95,16329.0,United Kingdom,1.95,False,False,False
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,5.10,False,False,False
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01,0.65,16329.0,United Kingdom,7.80,False,False,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,3.75,False,False,False


In [24]:
#Save a review file, then remove exact duplicates from the analysis copy

from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/day12_data_cleaning"
)
output_dir.mkdir(parents=True, exist_ok=True)

# Save all rows that belong to duplicate groups for review
duplicate_review.to_csv(
    output_dir / "exact_duplicate_records_for_review.csv",
    index=False
)

# Remove extra copies, keeping the first matching record
cleaned_sales_before_dedup = cleaned_sales.copy()

cleaned_sales = cleaned_sales.drop_duplicates(
    subset=original_columns,
    keep="first"
).copy()

print("Rows before deduplication:", len(cleaned_sales_before_dedup))
print("Rows after deduplication:", len(cleaned_sales))
print(
    "Exact duplicate copies removed:",
    len(cleaned_sales_before_dedup) - len(cleaned_sales)
)
print(
    "Remaining exact duplicate rows:",
    int(cleaned_sales.duplicated(subset=original_columns).sum())
)

Rows before deduplication: 1041670
Rows after deduplication: 1007911
Exact duplicate copies removed: 33759
Remaining exact duplicate rows: 0


In [25]:
#update the cleaning log

duplicate_copies_removed = (
    len(cleaned_sales_before_dedup) - len(cleaned_sales)
)

duplicate_log_entry = pd.DataFrame([{
    "Issue": "Exact duplicate transaction records",
    "Rows_Affected": duplicate_copies_removed,
    "Action_Taken": (
        "Saved all rows in duplicate groups for review; "
        "kept the first copy of exact matches across the eight "
        "original transaction columns in the analysis dataset."
    )
}])

display(duplicate_log_entry)

,Issue,Rows_Affected,Action_Taken
0,Exact duplicate transaction records,33759,Saved all rows in duplicate groups for review;...


In [26]:
duplicate_log_entry.to_csv(
    output_dir / "duplicate_cleaning_log.csv",
    index=False
)

In [27]:
#return the five automated quality checks

quality_checks = pd.DataFrame([
    {
        "Check": "1. Completeness",
        "Test": "No missing dates in cleaned sales",
        "Passed": bool(cleaned_sales["InvoiceDate"].notna().all()),
        "Details": (
            f"Missing dates: "
            f"{cleaned_sales['InvoiceDate'].isna().sum()}"
        )
    },
    {
        "Check": "2. Uniqueness",
        "Test": "No exact duplicate transaction records",
        "Passed": bool(
            cleaned_sales.duplicated(
                subset=original_columns
            ).sum() == 0
        ),
        "Details": (
            "Checks all eight original transaction columns; "
            f"remaining duplicates: "
            f"{cleaned_sales.duplicated(subset=original_columns).sum()}"
        )
    },
    {
        "Check": "3. Validity",
        "Test": "All quantities and prices are positive",
        "Passed": bool(
            cleaned_sales["Quantity"].gt(0).all()
            and cleaned_sales["Price"].gt(0).all()
        ),
        "Details": (
            f"Non-positive quantities: "
            f"{cleaned_sales['Quantity'].le(0).sum()}; "
            f"non-positive prices: "
            f"{cleaned_sales['Price'].le(0).sum()}"
        )
    },
    {
        "Check": "4. Consistency",
        "Test": "Revenue equals quantity multiplied by price",
        "Passed": bool(
            np.allclose(
                cleaned_sales["Revenue"],
                cleaned_sales["Quantity"] * cleaned_sales["Price"],
                equal_nan=False
            )
        ),
        "Details": "Compares Revenue with Quantity × Price"
    },
    {
        "Check": "5. Freshness",
        "Test": "Date range is valid and ordered",
        "Passed": bool(
            len(cleaned_sales) > 0
            and cleaned_sales["InvoiceDate"].min()
                <= cleaned_sales["InvoiceDate"].max()
        ),
        "Details": (
            f"From {cleaned_sales['InvoiceDate'].min()} "
            f"to {cleaned_sales['InvoiceDate'].max()}"
        )
    }
])

display(quality_checks)


,Check,Test,Passed,Details
0,1. Completeness,No missing dates in cleaned sales,True,Missing dates: 0
1,2. Uniqueness,No exact duplicate transaction records,True,Checks all eight original transaction columns;...
2,3. Validity,All quantities and prices are positive,True,Non-positive quantities: 0; non-positive price...
3,4. Consistency,Revenue equals quantity multiplied by price,True,Compares Revenue with Quantity × Price
4,5. Freshness,Date range is valid and ordered,True,From 2009-12-01 00:00:00 to 2011-12-09 00:00:00


In [28]:
#check the overall results

failed_checks = quality_checks[~quality_checks["Passed"]]

print("Total checks:", len(quality_checks))
print("Passed:", int(quality_checks["Passed"].sum()))
print("Failed:", len(failed_checks))

if len(failed_checks) > 0:
    print("\nChecks needing further investigation:")
    display(failed_checks)
else:
    print("\nAll five automated quality checks passed.")

Total checks: 5
Passed: 5
Failed: 0

All five automated quality checks passed.


In [29]:
#refresh the cleaning summary and save all the final files

cleaning_summary = pd.DataFrame([
    {"Metric": "Original combined rows", "Count": len(raw_df)},
    {
        "Metric": "Cancellation invoice rows",
        "Count": int(clean_df["Is_Cancellation_Invoice"].sum())
    },
    {
        "Metric": "Negative quantity rows",
        "Count": int(clean_df["Is_Negative_Quantity"].sum())
    },
    {
        "Metric": "Missing customer ID rows",
        "Count": int(clean_df[customer_col].isna().sum())
    },
    {
        "Metric": "Exact duplicate rows in original data",
        "Count": int(raw_df.duplicated().sum())
    },
    {
        "Metric": "Invalid date rows",
        "Count": int(clean_df["InvoiceDate"].isna().sum())
    },
    {
        "Metric": "Missing quantity rows",
        "Count": int(clean_df["Quantity"].isna().sum())
    },
    {
        "Metric": "Missing price rows",
        "Count": int(clean_df["Price"].isna().sum())
    },
    {
        "Metric": "Exact duplicate copies removed from cleaned sales",
        "Count": duplicate_copies_removed
    },
    {
        "Metric": "Final cleaned sales rows",
        "Count": len(cleaned_sales)
    },
    {
        "Metric": "Potential revenue outlier rows flagged",
        "Count": int(cleaned_sales["Is_Revenue_Outlier"].sum())
    }
])

display(cleaning_summary)

,Metric,Count
0,Original combined rows,1067371
1,Cancellation invoice rows,19494
2,Negative quantity rows,22950
3,Missing customer ID rows,243007
4,Exact duplicate rows in original data,12135
5,Invalid date rows,0
6,Missing quantity rows,0
7,Missing price rows,0
8,Exact duplicate copies removed from cleaned sales,33759
9,Final cleaned sales rows,1007911


In [30]:
#save the final outputs

# Save the final cleaned analysis dataset
cleaned_sales.to_csv(
    output_dir / "cleaned_sales.csv",
    index=False
)

# Save the refreshed summary and quality checks
cleaning_summary.to_csv(
    output_dir / "cleaning_summary.csv",
    index=False
)

quality_checks.to_csv(
    output_dir / "five_quality_checks.csv",
    index=False
)

# Save supporting records for review
clean_df[clean_df["Is_Cancellation_Invoice"]].to_csv(
    output_dir / "cancellation_records.csv",
    index=False
)

clean_df[clean_df["Is_Negative_Quantity"]].to_csv(
    output_dir / "negative_quantity_records.csv",
    index=False
)

cleaned_sales[cleaned_sales["Is_Revenue_Outlier"]].to_csv(
    output_dir / "revenue_outliers_for_review.csv",
    index=False
)

print("Final files saved to:", output_dir)
print("\nFiles in output folder:")
for file_path in sorted(output_dir.glob("*.csv")):
    print("-", file_path.name)

Final files saved to: /content/drive/MyDrive/MVJ_Internship/Project1/day12_data_cleaning

Files in output folder:
- cancellation_records.csv
- cleaned_sales.csv
- cleaning_summary.csv
- duplicate_cleaning_log.csv
- exact_duplicate_records_for_review.csv
- five_quality_checks.csv
- negative_quantity_records.csv
- revenue_outliers_for_review.csv


In [31]:
#final verification before packing up

print("FINAL DAY 12 VERIFICATION")
print("-" * 35)

print("Original dataset rows:", len(raw_df))
print("Final cleaned sales rows:", len(cleaned_sales))
print("Missing dates:", int(cleaned_sales["InvoiceDate"].isna().sum()))
print(
    "Duplicate transaction records:",
    int(cleaned_sales.duplicated(subset=original_columns).sum())
)
print("Quality checks passed:", int(quality_checks["Passed"].sum()), "/ 5")
print("Quality checks failed:", int((~quality_checks["Passed"]).sum()))

print("\nOutput folder exists:", output_dir.exists())
print("Cleaned sales CSV exists:", (output_dir / "cleaned_sales.csv").exists())
print("Quality checks CSV exists:", (output_dir / "five_quality_checks.csv").exists())

FINAL DAY 12 VERIFICATION
-----------------------------------
Original dataset rows: 1067371
Final cleaned sales rows: 1007911
Missing dates: 0
Duplicate transaction records: 0
Quality checks passed: 5 / 5
Quality checks failed: 0

Output folder exists: True
Cleaned sales CSV exists: True
Quality checks CSV exists: True


DONE WITH DAY 12 TASKS